# 训练型检索器扩展：让检索器适应回答模型

普通 RAG 往往分别训练检索器和回答模型。但“语义相似”的资料不一定是回答模型最会使用的资料。PRCA（在检索器和回答模型之间重新组织片段）和 REPLUG（用回答模型偏好调整检索器）都试图缩小这个差距。

本页是训练型扩展阅读：保留方法说明和训练目标的可运行小例子，不调用回答模型，也没有训练 PRCA 或 REPLUG；输出不是这两种方法的效果比较。基础 RAG 的问题筛选、改写、拆题和继续检索，请先阅读本章前 5 个 Notebook；真实的向量模型微调请看 C2。

- PRCA 在检索结果和回答模型之间加一个小型调整模块，为不同片段重新分配作用。
- REPLUG 保持回答模型不变，使用回答模型对文本的偏好来调整检索器。


## 本页导航与使用边界

1. 基础路径：先用筛选、改写、拆题和继续检索解决普通召回问题。
2. 训练型扩展：只有当检索器需要适应回答模型偏好、且手上有输入—答案或上下文标注时，才阅读 PRCA/REPLUG。
3. 本页示意：softmax 单元只展示两个权重分布如何对齐，不是训练过程、模型输出或质量提升结果。

如果要做真实的向量模型微调，进入[什么时候需要微调向量模型](../2.%20数据处理/什么时候需要微调向量模型.ipynb)，不要把下面的分布示意当成已经完成的微调。

## 两条路线：加 Adapter，或微调检索器

检索器通常按向量相似度返回资料，但回答模型真正容易使用的资料可能更短、更可读，或正好包含回答所需的关系。原论文综述把对齐分为两类：第一类在检索器和生成器之间加入 Adapter（适配器），从返回的前几条资料中抽取、重组或加权上下文；第二类用大模型的反馈微调检索器，让检索分布更接近回答模型偏好的分布。两类都需要训练和独立评估，不是加一条提示词就完成。相关方法还包括用于提取式/生成式压缩的 RECOMP、把知识整合进白盒模型的 PKG、用 FiD 交叉注意力提供监督的 AAR，以及从提示池检索任务提示的 UPRISE；本页重点展开 PRCA 和 REPLUG。

### PRCA：插在检索器和生成器之间的奖励驱动适配器

PRCA（可插拔奖励驱动上下文适配器）把一个轻量模块放在检索器与回答模型之间，生成器可以保持冻结，甚至可以是只能通过 API 调用的黑盒模型。它接收用户问题和检索到的前几条文档，学习筛选、重组或加权其中的信息；在生成过程中还可以按 token 自回归地调整上下文表示。优化目标是用策略梯度把最终答案质量变成奖励，让适配器提高期望奖励，而不是只追求嵌入相似度。

![PRCA 架构](./figures/PRCA-1.png)

PRCA 分两个阶段训练。第一阶段是上下文提取（Contextual Extraction Stage）：输入由问题和检索器返回的前几条文档组成，适配器生成上下文 C_extracted，并用真实上下文 C_truth 监督，目标可写成 `min L(θ) = −(1/N) Σ C_truth log f_PRCA(S_input; θ)`。第二阶段是奖励驱动（Reward-Driven Stage）：把适配器生成的上下文送进生成器，用生成答案与真实答案的差异作为反馈。示例使用 ROUGE-L 作为奖励，再用策略梯度更新适配器。

![PRCA 两阶段训练](./figures/PRCA-2.png)

PRCA 适合已有检索器、但需要适配多个回答任务或只能访问黑盒回答模型的场景；它的好处是不用改生成器，缺点是需要带答案或上下文标注的训练数据、奖励设计和 rollout，训练成本包含多次生成调用及策略梯度的不稳定性，线上还会增加适配器的延迟。若只是资料排序错误或没有训练数据，应先用本章的改写、重排和压缩方法，不应把当前小例子的 softmax 权重当成 PRCA 已训练的效果。

### REPLUG：用语言模型监督检索器

REPLUG 是增强黑盒语言模型的外部检索框架，不访问回答模型的内部参数，也不要求微调回答模型。给定输入上下文 x，双编码器检索器在外部语料库 D 中按嵌入相似度找前几条文档 D′。对每个文档 d，将 d 拼到 x 前面形成 d ∘ x，分别让语言模型预测，再按检索相似度权重混合输出概率：`p(y | x, D′) = Σ p(y | d ∘ x) · λ(d, x)`，其中 `λ(d, x) = exp(s(d,x)) / Σ exp(s(d,x))`。

![REPLUG 输入重构](./figures/REPLUG-1.png)

REPLUG LSR（LM-Supervised Retrieval）进一步用语言模型监督检索器。训练时先计算检索概率 PR(d | x)，再计算给定目标答案 y 时语言模型对每个候选文档的概率 Q_LM(d | x, y)，把两者的 KL 散度作为损失：`L = (1/|B|) Σ KL(PR(d | x) ∥ Q_LM(d | x, y))`。检索器因此会偏向降低语言模型困惑度、真正有助于预测答案的文档；训练期间每隔 T 个步骤重新计算文档嵌入并更新索引，以免索引停留在旧参数上。

![REPLUG 的概率混合](./figures/REPLUG-2.png)

![REPLUG LSR 训练](./figures/REPLUG-3.png)

REPLUG 适合不能改动商业 API、但可以维护外部语料和检索器的场景；它的优点是回答模型保持黑盒，缺点是每个候选文档都需要语言模型打分或前向计算，推理成本随 K 增长。LSR 训练还需要输入—答案数据、冻结语言模型的监督概率、检索器更新和周期性索引重建，显存、计算时间和索引维护成本都明显高于普通 BM25/向量检索。当前代码只演示两个分布如何对齐，未训练 PRCA 或 REPLUG，也没有提供实际问题上的提升结论。

参考论文：[PRCA](https://arxiv.org/abs/2310.18347)、[REPLUG](https://arxiv.org/abs/2301.12652)，以及方法综述 [Aligning Retrieval with LLM Preferences](https://arxiv.org/abs/2312.10997)。

In [1]:
from math import exp

retriever_scores = [1.2, 0.7, 0.2]
answer_model_scores = [0.4, 1.1, 0.1]

def softmax(values):
    total = sum(exp(value) for value in values)
    return [exp(value) / total for value in values]

before = softmax(retriever_scores)
target = softmax(answer_model_scores)
print("检索器原权重：", [round(value, 3) for value in before])
print("回答模型更容易使用的权重：", [round(value, 3) for value in target])
print("训练目标示意：让检索权重逐步接近第二组分布；本单元没有执行训练。")

检索器原权重： [0.506, 0.307, 0.186]
回答模型更容易使用的权重： [0.266, 0.536, 0.197]
训练目标示意：让检索权重逐步接近第二组分布；本单元没有执行训练。


这两种方法需要训练数据、足够计算资源和独立评估，不是一段提示词就能完成的调整。本页保留原理和训练目标；真正训练 PRCA 或 REPLUG 的成本不适合在本页启动，因此不声称已经提高了教程问题的回答质量。

## 本页导航

- 章节入口：[本章 README](README.md)
- 运行准备：[C7 统一运行准备](../README.md#运行准备)
- 相关下一步：[进入生成阶段](../5.%20生成阶段/排序、压缩与回答.ipynb)

